# Phase 2 : Ingénierie des données - Data Cleaning

## Introduction
Suite à notre Analyse Exploratoire des Données (EDA), nous avons identifié plusieurs axes de nettoyage nécessaires pour préparer notre jeu de données :
1. Traitement des valeurs manquantes réelles (`NaN`) et des valeurs mal renseignées (comme les "no data" dans les types de sols).
2. Correction des types de données (notamment la variable `Taux_5b_hydro` reconnue comme du texte à cause des virgules, ainsi que les dates).
3. Gestion des valeurs extrêmes identifiées sur notre variable cible `Taux_Chlordecone`.

---

### 1. Importation des données et diagnostic des valeurs manquantes (NA)
Pour commencer, nous importons notre jeu de données brut. La première étape de nettoyage consiste à quantifier exactement les valeurs manquantes pour chaque colonne afin de définir notre stratégie (imputation ou suppression).

In [1]:
import pandas as pd

#1.chargement des données
df = pd.read_csv('BaseCLD2026.csv', sep=';')
df.head()

,ID,ANNEE,COMMU_LAB,RAIN,Sol_simple,type_sol,Date_prelevement,Date_enregistrement,Date_analyse,Operateur_chld,...,Taux_5b_hydro,histoBanane_Histo_ban,mnt_tpi_mean,mnt_tri_mean,mnt_rugosite_mean,mnt_ombrage_mean,mnt_exposition_mean,mnt_pente_mean,X,Y
0,20143,2010,GROS-MORNE,2000-3000,Andosol,Intergrades Sols … allophane relativement ‚vol...,24/05/2007,24/05/2007,24/05/2007,=,...,"0,07",2.0,5.805967,8.033367,21.593658,131.173998,79.447954,39.043098,714300.831892,1.626344e+06
1,20143,2010,GROS-MORNE,2000-3000,Andosol,Intergrades Sols … allophane relativement ‚vol...,24/05/2007,24/05/2007,24/05/2007,=,...,"0,07",2.0,5.683589,7.920563,20.944901,134.608205,76.985690,38.123675,714303.743345,1.626354e+06
2,20143,2010,GROS-MORNE,2000-3000,Andosol,Intergrades Sols … allophane relativement ‚vol...,24/05/2007,24/05/2007,24/05/2007,=,...,"0,07",3.0,2.239457,7.108432,20.085883,139.420523,76.064719,34.976678,714309.446765,1.626360e+06
3,20143,2010,GROS-MORNE,2000-3000,Andosol,Intergrades Sols … allophane relativement ‚vol...,24/05/2007,24/05/2007,24/05/2007,=,...,"0,07",1.0,4.038373,7.530090,23.427731,121.603678,92.392454,38.315796,714294.208512,1.626321e+06
4,20143,2010,GROS-MORNE,2000-3000,Andosol,Intergrades Sols … allophane relativement ‚vol...,24/05/2007,24/05/2007,24/05/2007,=,...,"0,07",2.0,0.596502,6.637082,20.153770,134.065066,83.930611,33.877727,714303.823058,1.626341e+06


In [2]:
#2.calcul du nombre de valeurs manquantes par colonne 
missing_values = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne:", missing_values)

Nombre de valeurs manquantes par colonne: ID                           0
ANNEE                        0
COMMU_LAB                  298
RAIN                         0
Sol_simple                  74
type_sol                  2609
Date_prelevement             0
Date_enregistrement          0
Date_analyse                 0
Operateur_chld               0
Taux_Chlordecone             0
Operateur_5b                 0
Taux_5b_hydro               12
histoBanane_Histo_ban    17983
mnt_tpi_mean                28
mnt_tri_mean                28
mnt_rugosite_mean           28
mnt_ombrage_mean            28
mnt_exposition_mean         28
mnt_pente_mean              28
X                            0
Y                            0
dtype: int64


In [3]:
#3.Pourceentage que cela represente par rapport au total des lignes 
missing_percentage = (missing_values / len(df)) * 100
print("Pourcentage de valeurs manquantes par colonne:", missing_percentage)

Pourcentage de valeurs manquantes par colonne: ID                        0.000000
ANNEE                     0.000000
COMMU_LAB                 0.957399
RAIN                      0.000000
Sol_simple                0.237743
type_sol                  8.382060
Date_prelevement          0.000000
Date_enregistrement       0.000000
Date_analyse              0.000000
Operateur_chld            0.000000
Taux_Chlordecone          0.000000
Operateur_5b              0.000000
Taux_5b_hydro             0.038553
histoBanane_Histo_ban    57.774851
mnt_tpi_mean              0.089957
mnt_tri_mean              0.089957
mnt_rugosite_mean         0.089957
mnt_ombrage_mean          0.089957
mnt_exposition_mean       0.089957
mnt_pente_mean            0.089957
X                         0.000000
Y                         0.000000
dtype: float64


In [4]:
#4.tableau récapitulatif pour y voir plus clair
missing_summary = pd.DataFrame({
    'Nombre de valeurs manquantes': missing_values,
    'Pourcentage de valeurs manquantes': missing_percentage
})

colonnes_avec_na = missing_summary[missing_summary
['Nombre de valeurs manquantes'] > 0].sort_values(by='Nombre de valeurs manquantes', ascending=False)

print("Colonnes avec des valeurs manquantes:", colonnes_avec_na)

Colonnes avec des valeurs manquantes:                        Nombre de valeurs manquantes  \
histoBanane_Histo_ban                         17983   
type_sol                                       2609   
COMMU_LAB                                       298   
Sol_simple                                       74   
mnt_tpi_mean                                     28   
mnt_tri_mean                                     28   
mnt_rugosite_mean                                28   
mnt_exposition_mean                              28   
mnt_ombrage_mean                                 28   
mnt_pente_mean                                   28   
Taux_5b_hydro                                    12   

                       Pourcentage de valeurs manquantes  
histoBanane_Histo_ban                          57.774851  
type_sol                                        8.382060  
COMMU_LAB                                       0.957399  
Sol_simple                                      0.237743  
mnt_tp

### 2. Stratégie de Data Cleaning
Le diagnostic des valeurs manquantes nous révèle trois situations distinctes :
* **Forte proportion (> 50%)** : `histoBanane_Histo_ban` (57.8%). Une imputation classique risque de biaiser les données. Il faudra décider si l'on supprime la variable ou si l'on remplace les NA par une valeur par défaut (ex: 0, en supposant que l'absence d'information signifie l'absence de bananeraie).
* **Proportion modérée (~8%)** : `type_sol` (8.4%). Nous procéderons à une imputation (remplacement par le mode ou imputation algorithmique).
* **Proportion faible (< 1%)** : Variables topographiques (`mnt_...`), `COMMU_LAB`, `Sol_simple` et `Taux_5b_hydro`. Les valeurs manquantes étant infimes, une imputation simple (médiane/mode) ou une suppression des lignes suffira.

---

### 3. Correction des formats de données et traitement des anomalies textuelles
Lors de la conversion de la variable `Taux_5b_hydro` (taux numérique contenant des virgules), nous avons identifié la présence de chaînes de caractères imprévues (ex: "détecté"). 

Pour nettoyer cette variable sans bloquer le pipeline, nous utilisons une conversion sécurisée (`pd.to_numeric` avec l'option `coerce`). 

In [5]:
#1.remplacement de la virgule par un point pour la colonne Taux_5b_hydro
df['Taux_5b_hydro'] = df['Taux_5b_hydro'].astype(str).str.replace(',', '.')

In [6]:
#2. Conversion en numerique en forçant les textes non convertibles en NaN
df['Taux_5b_hydro'] = pd.to_numeric(df['Taux_5b_hydro'], errors='coerce')

In [7]:
#3. verification du type et du nouveau nombre de NA 
print("Type de la colonne Taux_5b_hydro:", df['Taux_5b_hydro'].dtype)
print("Nombre de valeurs manquantes dans Taux_5b_hydro après conversion:", df['Taux_5b_hydro'].isnull().sum())

Type de la colonne Taux_5b_hydro: float64
Nombre de valeurs manquantes dans Taux_5b_hydro après conversion: 23


### 4. Traitement des valeurs manquantes (Imputation et Suppression)
En nous basant sur notre diagnostic et sur le contexte agronomique de l'étude, voici la stratégie appliquée pour traiter les valeurs manquantes :

1. **`histoBanane_Histo_ban`** : Cette variable est cruciale car la chlordécone était principalement utilisée sur les bananeraies. Les valeurs manquantes (58%) correspondent très probablement à des parcelles n'ayant jamais cultivé de bananes. Nous les imputons donc par la valeur **0**.
2. **`type_sol`** : Avec ~8% de valeurs manquantes, nous optons pour une imputation statistique classique en remplaçant les vides par la modalité la plus fréquente (le **Mode**).
3. **Autres variables (< 1%)** : Pour les variables topographiques, `COMMU_LAB` et `Taux_5b_hydro`, la proportion de valeurs manquantes est dérisoire. Nous pouvons supprimer ces quelques lignes restantes sans risquer de perdre de l'information représentative

In [8]:
#1 Imputation de histoBanane_Histo_ban par 0
df['histoBanane_Histo_ban'] = df['histoBanane_Histo_ban'].fillna(0)

In [9]:
#2. Imputation de type_sol par le mode
mode_type_sol = df['type_sol'].mode()[0]
df['type_sol'] = df['type_sol'].fillna(mode_type_sol)

In [10]:
#3. suppression des lignes restantes avec des NA pour les colonnes à <1%
df = df.dropna()

In [11]:
#4. verification finale de la taille du dataset et des NA restants
print("Nombre total de valeurs manquantes après nettoyage:", df.isnull().sum().sum())
print("Taille du dataset après nettoyage:", df.shape)

Nombre total de valeurs manquantes après nettoyage: 0
Taille du dataset après nettoyage: (30708, 22)


In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 30708 entries, 0 to 31092
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     30708 non-null  int64  
 1   ANNEE                  30708 non-null  int64  
 2   COMMU_LAB              30708 non-null  str    
 3   RAIN                   30708 non-null  str    
 4   Sol_simple             30708 non-null  str    
 5   type_sol               30708 non-null  str    
 6   Date_prelevement       30708 non-null  str    
 7   Date_enregistrement    30708 non-null  str    
 8   Date_analyse           30708 non-null  str    
 9   Operateur_chld         30708 non-null  str    
 10  Taux_Chlordecone       30708 non-null  float64
 11  Operateur_5b           30708 non-null  str    
 12  Taux_5b_hydro          30708 non-null  float64
 13  histoBanane_Histo_ban  30708 non-null  float64
 14  mnt_tpi_mean           30708 non-null  float64
 15  mnt_tri_mean      

In [13]:
df.describe()

C:\Users\mopit\AppData\Roaming\Python\Python314\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,ID,ANNEE,Taux_Chlordecone,Taux_5b_hydro,histoBanane_Histo_ban,mnt_tpi_mean,mnt_tri_mean,mnt_rugosite_mean,mnt_ombrage_mean,mnt_exposition_mean,mnt_pente_mean,X,Y
count,30708.000000,30708.000000,30708.000000,30708.0000,30708.000000,30708.000000,30708.000000,30708.000000,30708.000000,30708.000000,30708.000000,30708.000000,3.070800e+04
mean,38638.665006,2015.313436,0.669486,inf,0.709457,0.210060,4.323354,13.472464,174.893656,169.749789,20.316921,713037.410135,1.624052e+06
std,19418.047728,2.818994,1.567425,NaN,0.955515,2.585794,2.859913,8.618081,27.922202,95.575834,13.367818,9874.541713,1.176078e+04
min,20003.000000,2010.000000,0.001000,0.0000,0.000000,-29.065807,0.000000,0.000000,24.789383,0.000000,0.000000,690703.409647,1.594393e+06
25%,21861.000000,2013.000000,0.002300,0.0010,0.000000,-0.875000,2.389537,7.539842,159.211437,89.531867,10.871326,703247.895574,1.614078e+06
50%,30309.000000,2016.000000,0.003300,0.0033,0.000000,0.118970,3.833333,12.000000,177.000000,168.740226,17.947773,713493.233694,1.624495e+06
75%,62802.000000,2018.000000,0.403000,0.0150,1.000000,1.266916,5.625000,17.798259,193.178707,248.629379,27.120739,721000.025534,1.633391e+06
max,63714.000000,2019.000000,17.350000,inf,3.000000,26.011950,29.358208,85.396165,255.000000,359.144897,124.949989,734338.584306,1.645435e+06


### 5. Traitement des valeurs infinies (Anomalies post-conversion)
Suite à l'observation de nos statistiques descriptives post-nettoyage, nous avons détecté une anomalie critique sur la variable `Taux_5b_hydro` : sa moyenne et son maximum valent `inf` (l'infini). 

Cela est dû à la présence de valeurs textuelles interprétées comme des infinis mathématiques lors de la conversion. Pour éviter de faire échouer nos futurs modèles statistiques (comme l'Analyse en Composantes Principales ou les matrices de corrélations), nous allons remplacer ces valeurs infinies par des valeurs manquantes (`NaN`) puis supprimer ces quelques lignes résiduelles.

In [14]:
import numpy as np

# 1. Remplacement des infinis par des NaN (réassignation classique, plus sûre)
df = df.replace([np.inf, -np.inf], np.nan)

# 2. Suppression de ces nouvelles valeurs manquantes
df = df.dropna()

# 3. Vérification que l'infini a bien disparu de Taux_5b_hydro
print(df['Taux_5b_hydro'].describe())

count    30129.000000
mean         4.593618
std         67.455172
min          0.000000
25%          0.001000
50%          0.003300
75%          0.015000
max        999.000000
Name: Taux_5b_hydro, dtype: float64


In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 30129 entries, 0 to 31092
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     30129 non-null  int64  
 1   ANNEE                  30129 non-null  int64  
 2   COMMU_LAB              30129 non-null  str    
 3   RAIN                   30129 non-null  str    
 4   Sol_simple             30129 non-null  str    
 5   type_sol               30129 non-null  str    
 6   Date_prelevement       30129 non-null  str    
 7   Date_enregistrement    30129 non-null  str    
 8   Date_analyse           30129 non-null  str    
 9   Operateur_chld         30129 non-null  str    
 10  Taux_Chlordecone       30129 non-null  float64
 11  Operateur_5b           30129 non-null  str    
 12  Taux_5b_hydro          30129 non-null  float64
 13  histoBanane_Histo_ban  30129 non-null  float64
 14  mnt_tpi_mean           30129 non-null  float64
 15  mnt_tri_mean      

### 6. Conclusion du Data Cleaning
Le nettoyage de nos données est terminé. Nous avons traité avec succès les valeurs manquantes, corrigé les erreurs de format (variables textuelles intruses) et éliminé les valeurs mathématiques bloquantes (infinis). Notre jeu de données final, sain et robuste, contient **30 129 observations et 22 variables**.

---

In [16]:
# Sauvegarde du jeu de données nettoyé
df.to_csv('chlordecone_clean.csv', index=False)